In [5]:
# Décommenter la ligne suivante pour installer les dépendances
#%pip install jupyter_bokeh nbconvert panel watchfiles

In [6]:
import matplotlib.pyplot as plt
import pandas as pd
import panel as pn

pd.set_option('display.max_rows', 500)
import numpy as np


import ast
import json

pn.extension()

In [ ]:
# importation des images
histo_y = pn.pane.Image("/home/onyxia/work/FORMATION-DS/PROJET/images_panel/3.1.histogramme_notes_moyennes.jpeg", width=1000)
histo_vote_count = pn.pane.Image("/home/onyxia/work/FORMATION-DS/PROJET/images_panel/3.2.histogramme_vote_count.jpeg", width=1000)
graph_films_votes = pn.pane.Image("/home/onyxia/work/FORMATION-DS/PROJET/images_panel/3.3.graph_films_votes.jpeg", width=1200)
graph_films_notes = pn.pane.Image("/home/onyxia/work/FORMATION-DS/PROJET/images_panel/3.4.graph_nb_films_notes_moyennes.jpeg", width=1000)


In [ ]:

text_y_hist = pn.pane.Markdown("""
la moyenne de la distribution de vote_average est 6.09. Les observations sont assez resserrées autour de la moyenne, l'écart-type est de 1.07
""")
text_vote_count_hist = pn.pane.Markdown("""
Le nombre de votes par film prend des valeurs de 1 jusqu'a 14 075 (d'où l'utilité de prendre le logarithme), avec une moyenne de 118,7. 
Alerte : 25 % des films ont moins de 5 votes. 
""")
texte_gr_films_votes =pn.pane.Markdown("""
avant 1940 peu de films
a partir de 1940 : le nombre de films par an dépasse la centaine et augmente de manière exponentielle à partir de 1990 (création d’IMDB)
En retirant tous les films qui ont moins de 5 votes, on s'assure d'avoir de notes moyennes qui ont plus de sens
Par ailleurs nous utiliserons vote_count comme pondération dans nos modèles 

NB : pic en 1902 : il n’y a que 2 films : 
1 qui n’a que 4 votes donc qui n’est pas dans le graphique de droite
et l’autre qui a 314 votes : il s’agit du “ voyage dans la lune” (film de science fiction de George Meliès)
""")   
texte_gr_films_notes = pn.pane.Markdown("""
compromis : nous disposons de moins de films , mais de notes moins volatiles (avant 1950 notamment)

Contrairement à ce qu’on pourrait croire, en retirant les films avec moins de 5 votes, les notes moyennes augmentent  
(les films avec peu de votes n’étaient pas notés par le meilleur ami du réalisateur mais plutôt par sa belle-mère)

À partir de 1940 la moyenne des notes est plutôt sur une tendance décroissante (sauf toute fin de la série soit 2017).

C’est compatible avec l’idée que les films antérieurs à 1990 qu’on a pris la peine d’ajouter sur IMDB sont ceux déjà devenus cultes → biais de sélection à la hausse?  
Ou tout simplement public plus indulgent ?

C’est notamment au vu de la distribution des films dans le temps que nous renonçons à considérer une dimension temporelle dans notre modèle pour prédire Y.  
On considère que les films sont indépendants (et a fortiori les films des différentes années).  
Exception pour les sagas : on a bien tenu compte d’une info temporelle avec la note de l’opus précédent comme variable explicative.
""")


In [ ]:
# Titre principal
section_title = pn.pane.Markdown("# 4. Machine Learning")

# Sous-partie 1 : Modèles linéaires
reg_title = pn.pane.Markdown("## Modèles linéaires")

reg0_title = pn.pane.Markdown("### Modèle 0 (OLS)")
reg0_text = pn.pane.Markdown(""" modèle qui sert de benchmark avec 47 variables explicatives
l'entraînement sur 90 % du dataset (27 500 observations)
et 10 % en échantillon test 
indicateurs de performance du modèle :
pour chaque modèle, on va garder le MAE , le RMSE et le R² 
mais le critère principal reste le RMSE
""")
reg0_img = pn.pane.Image("/home/onyxia/work/FORMATION-DS/PROJET/images_panel/4.0.results_ols.jpeg", width=(350))

reg1_title = pn.pane.Markdown("### Modèle 1 WLS")
reg1_text = pn.pane.Markdown(""" L’idée est de tenir compte du fait qu’une note moyenne basée sur 5 avis ne doit pas avoir le meme poids dans notre apprentissage qu’une note basée sur 10000 avis.
Nous pondérons nos observations d’apprentissage (X_train) par log(vote_count) (le nombre de votes conduisant à la note moyenne du film)

Contrairement à ce qu’on pensait, les performances du modèle pondéré WLS est pire sur les 3 critères

.""")
reg1_img = pn.pane.Image("/home/onyxia/work/FORMATION-DS/PROJET/images_panel/4.1.results_wls.jpeg", width=(350))

reg2_title = pn.pane.Markdown("### Modèle 2 BACKWARD")
reg2_text = pn.pane.Markdown(""" Notre modèle de départ WLS utilise  47 variables explicatives (+ la constante). Une piste explorée pour améliorer le modèle : mettre en oeuvre un algorithme backward de sélection de variables.
le critère adopté est l’AIC.

le modèle final supprime 12 variables :
Asia
Latin_America
Africa
Europe
WarWestern
ThrillerCrime
small_awards_before_film
big_awards_before_film_1
big_awards_before_film_2
big_awards_before_film_4
big_awards_before_film_7
small_awards_before_film_9

4 variables géographiques sur 6
2 modalités de genres sur 10
et surtout 6 variables récompenses sur 22, dont 3 concernant les récompenses du réalisateur et des 2 acteurs principaux

les performances restent moins bonnes que notre modèle de base OLS
""")
reg2_img = pn.pane.Image("/home/onyxia/work/FORMATION-DS/PROJET/images_panel/4.2.results_backward.jpeg", width=(350))

reg3_title = pn.pane.Markdown("### Modèle 3 ridge")
reg3_text = pn.pane.Markdown(""" La régression pénalisée Ridge peut peut être s'avérer utile , puisqu’elle sert quand les variables explicatives sont fortement corrélées
Pour ce modèle, le lambda est optimisé par validation croisée sur 10 blocs, avec comme critère -MSE

Les performances predictives du modèle sont toujours moins bonnes que le modèle OLS
""")
reg3_img = pn.pane.Image("/home/onyxia/work/FORMATION-DS/PROJET/images_panel/4.3.results_ridge.jpeg", width=(350))

reg4_title = pn.pane.Markdown("### Modèle 4 lasso")
reg4_text = pn.pane.Markdown(""" Toujours avec l’idée que nous avons peut être trop de variables, nous essayons un modèle de régression pénalisée Lasso.
Nous effectuons la même optimisation de lambda par VC 10 blocs
Dans le modèle final, une seule variable est supprimée 
big_awards_before_film_9 : c’est à dire le nombre de “grosses” récompenses obtenues ( avant le film) par le 9e acteur principal

Les performances du modèle toujours moins bonnes que le modèle  OLS
 """)
reg4_img = pn.pane.Image("/home/onyxia/work/FORMATION-DS/PROJET/images_panel/4.4.results_lasso.jpeg", width=(350))

reg5_title = pn.pane.Markdown("### Modèle 5 elasticnet")
reg5_text = pn.pane.Markdown(""" La régression elasticnet constitue notre ultime tentative sur des modèles linéaires
comme précédent, l’optimisation de λ est effectuée par validation croisée 10 blocs
le paramètre alpha est de 0,5
Encore une fois, les performances predictives du modèle sont moins bonnes qu’OLS
""")
reg5_img = pn.pane.Image("/home/onyxia/work/FORMATION-DS/PROJET/images_panel/4.5.results_elasticnet.jpeg", width=(350))

reg6_title = pn.pane.Markdown("### Modèle 6 LOG_BUDGET")
reg6_text = pn.pane.Markdown(""" Nous avons tenté une transformation de la variable du budget en log
car elle prend ses valeurs entre -1 (toutes les valeurs manquantes) jusqu’a plusieurs milliards de $
Remarque : Nous avons conscience des limites des retraitements effectués sur la variable : les plus gros budgets ( en milliards !) sont essentiellement des très vieux films pour lesquels les conversions de monnaies et le traitement pour passer en $ constants sont sans doute inadaptés (taux de conversion actuel vs conversion à l’époque, indice des prix seulement à partir de 1960…)

Voici les résultats pour l’ensemble des modèles linéaires précédemment présentés : 
La transformation en log améliore un peu le MAE dasn 4 modèles (WLS et les 3 regressions pénalisées) mais pas du tout le RMSE (notre critère principal) ni le R² : abandonné
""")
reg6_img = pn.pane.Image("/home/onyxia/work/FORMATION-DS/PROJET/images_panel/4.6.results_log_budget.jpeg", width=(350))

reg7_title = pn.pane.Markdown("### Modèle 7 erreurs pondérées")
reg7_text = pn.pane.Markdown(""" Dans nos observations X_train, nous donnons plus de poids aux films qui ont le plus de votes.
l’idée, c’est de pondérer aussi l’erreur avec la variable vote_count de notre échantillon test.
Ce n’est bien sûr pas pertinent en termes de prévisions, car nous ne disposerons jamais du nombre de votes pour prédire la note d’un film
Mais c’est pour confirmer ou infirmer que nous nous trompons plus sur les films qui ont peu de votes , mais qu’on prédit mieux les films avec une note moyenne plus robuste

et oui effectivement, les résultats sont tous significativement meilleurs
l’idéal aurait sans doute  été de fixer un seuil de vote_count plus haut (on a pris >=5) mais c’etait un compromis pour ne pas perdre trop d’observations (on en a enlevé déjà 25%)

le meilleur RMSE c’est lasso : 0.941723 ( contre 1.023148 sans pondérer l’erreur) . Il se trouve que c’est egalement le meilleur MAE
""")
reg7_img = pn.pane.Image("/home/onyxia/work/FORMATION-DS/PROJET/images_panel/4.7.results_ponderation.jpeg", width=(350))

# Sous-partie 2 : Random Forest
#rf_title = pn.pane.Markdown("## Random Forest")
#rf_text = pn.pane.Markdown("""  a completer """)
#rf_img = pn.pane.Image("images_panel/rf_results.jpeg", width=400)

# Sous-partie 3 : Gradient Boosting
#gb_title = pn.pane.Markdown("## Gradient Boosting")
#gb_text = pn.pane.Markdown("""  a completer """)
#gb_img = pn.pane.Image("images_panel/gb_results.jpeg", width=400)

# Sous-partie 4 : Réseaux de neurones
#nn_title = pn.pane.Markdown("## Réseaux de neurones")
#nn_text = pn.pane.Markdown("""  a completer """)
#nn_img = pn.pane.Image("images_panel/nn_results.jpeg", width=400)



In [ ]:


# ---- Contenu de chaque section ----
section_1 = pn.pane.Markdown("""
# 1. Introduction
Dans ce projet, nous avons décidé de nous intéresser à la base de données movies_metatadata.csv trouvable sur Kaggle, qui contient diverses informations sur 45 466 films sortis avant juillet 2017 (lien de la base kaggle). Cette base de données contient les variables suivantes :

- adult : indique si le film est "pour adultes"
- belongs to collection : si le film appartient à une série de films, la variable contient des informations sur la saga
- budget : budget du film
- genres : genres du film
- homepage : lien vers le site officiel du film s'il y en a un
- id : identifiant du film dans la base de données
- imdb_id : identifiant IMDB du film
- original_language : langue originale du film
- original_title : titre original du film
- overview : résumé du film
- popularity : popularité du film sur IMDB
- poster_path : lien vers l'affiche du film
- production_countries : pays de production du film
- production_companies : compagnies/maisons de production du film
- release_date : date de sortie du film
- revenue : recettes du film
- runtime : durée du film
- spoken_languages : langues parlées dans le film en version originale
- status : si le film est sorti, prévu, annulé, en production etc
- tagline : catchphrase du film
- title : titre anglophone du film
- video : False si le film est sorti au cinéma, True s'il est sorti directement sur Internet et qu'il n'a pas été diffusé au cinéma

Nous complétons ces informations avec la base de données credits.csv trouvée au même endroit et qui contient :
- id : identifiant du film dans la base de données
- cast : casting du film 
- crew : équipe du film 

Nous souhaitons à partir de ces données prédire la note de nouveaux films, voir quelles sont les variables les plus décisives pour prédire si un film sera bien reçu par le public et ainsi remarquer (ou non) la prévisibilité du succès d'un film.
""")



section_2 = pn.pane.Markdown("""
# 2. Retraitements
Après première exploration des données movies_metadata, nous décidons d'enlever les variables suivantes : 

- adult : elle présente presque toujours la modalité False
- homepage : renseigne le lien vers le site officiel du film s'il y en a un
- overview : contient les résumés des films, ce qui pourrait être intéressant à exploiter mais n'est pas l'objet du projet
- popularity : correspond à la popularité du film sur IMDB au moment où les données ont été extraites. C'est donc une variable qui n'est pas statique et dont nous ignorons le mode de calcul. Nous n'avons pas souhaiter la prendre en compte
- poster_path : lien vers l'affiche du film
- original_language : langue originale du film, extremement corrélée aux pays de production et predominance de l'anglais (71% des films)
- spoken_languages : indique les langues parlées durant le film en version originale, pas utile dans notre projet
- tagline indique la catchphrase du film

Nous allons nous concentrer sur les films qui sont déjà sortis en salle (status = Released et video=False) et qui ont un nombre de vote sur IMDB supérieur strictement à 0. 
Nous enlevons aussi les films où la date de sortie ou l'identifiant IMDB sont manquants (cela représente une soixantaine de films).

cas particulier de la variable "revenue" : contient les recettes du film mais 84% des observations sont manquantes ou nulles
Par ailleurs, ce n'est pas une donnée statique, en tout cas pour les films récents puisque les recettes augementent au fur et a mesure que le film se maintient en salle, sort dans de nouveaux pays etc
c'est meme une donnée pas du tout disponible pour les films qui viennent de sortir.
Puisque nous souhaitions être capables de prédire la note d'un film a peine sorti, nous avons décidé d'exclure cette variable. 
Mais avec une problématique élargie (predire la note d'un film pas forcement recent), il aurait pu etre pertinent de la garder

Une fois les variables nommées ci-dessus enlevées et les filtres appliqués, la base de données movies_metadata contient 42 028 films et 14 variables.

Les données du fichier credits.csv  permettent d'ajouter les acteurs principaux et le réalisateur du film à notre jeu de données. 


## Retraitement de la variable genres

La variable genres, qui indique les différents genres associés au film, est elle aussi sous forme de liste de dictionnaires. 
Chaque film peut avoir entre 0 et 8 genres différents parmi 20 modalités possibles.
Nous avons regroupé en 10 catégories pour eviter les trop petites classes :
* drama
* comedy , family
* thriller, crime,
* romance
* action, adventure
* war, western
* horror, mystery
* science fiction, fantasy
* animation
* intello (documentary, music, history, foreign)

Cas particulier pour la modalités "TV movie" : ne constitue pas un genre, mais un support de diffusion.
on decide d'ignorer cette modalité et de prendre en compte les autres variables genres associés au film (il y en a toujours au moins 1)

Comme un film peut avoir (et a le plus souvent) plusieurs genres associés, ces indicatrices ne s'excluent pas mutuellement
ce n'est pas a proprement parlé du "one hot encoding"


## Retraitement de la variable belongs_to_collection

si le film appartient à une série, cette variable contient des informations sur la saga sous forme de liste de dictionnaires. 
cela concerne initialement 4407 films. Pour les autres, la variable est vide
Ce format necessite un retraitement. A partir de "belongs_to_collection" nous créons 3 variables :
- "indic_collec" qui vaut 1 si le film fait partie d'une saga et 0 sinon, 
- "rang" qui indique le rang du film dans la saga (selon sa date de sortie) 
- "vote_prec" qui contient la note du film précédent dans la saga.
REMARQUE : notre périmètre se limite aux films présents dans notre dataset; il se peut que tous les opus d'une série ne soient pas présents.


## Retraitement de la variable production_countries

Il s'agit des pays (co)producteur(s) du film sous la forme de dictionnaires.
158 pays distincts sont présents et un même film peut avoir jusqu’à 25 pays coproducteurs
/!\ les pays sont la plupart du temps triés par ordre alphabétique, cela nous empeche de considérer le pays 1 comme le pays principal
exemple : “Bridget Jones’ diary” aurait été considéré comme un film français ;)

Nous avons remplacé les noms de pays par leur continent en distinguant Amérique du Nord et Amérique Latine.
Puis nous avons créé une variable par continent, indiquant le nombre de pays de ce continent associés à chaque film.
pour reprendre l'exemple de Bridget jones'diary : europe = 3 et North_America = 1

## Retraitement de la variable production_companies

Il s'agit de la ou des maisons de production du film.
/!\ un même film peut avoir jusqu’à 26 maisons de production 
plus de 20 000 maisons de production sont présentes dans notre dataset
Nous souhaitions les regrouper selon leur taille (petites, moyennes, grandes) 
nous avons adopté comme proxy de la taille , le nombre d’occurrences dans nos données
/!\ Pour être puristes, nous avons effectué le comptage des occurrences uniquement sur l'échantillon d'apprentissage (90%)
et nous en avons déduit ce decoupage en 3 classes :
- les petites : celles  n’apparaissent qu’une fois (13 199)
- les moyennes : celles qui apparaissent entre 2 et 10 fois
- les grosses : + de 10 occurrences
parmi elles, evidemment, les grosses productions americaines
le record :  Warner Bros.(856 films sur l'echantillon train) 


## Retraitement de la variable budget
Il s'agit du budget du film exprimé en $
Alerte : dans les données initiales,  33 265 lignes où le budget = 0 (soit pres de 80 % des films )
Nous avons tenté de recupérer l'information par webscraping sur wikipedia.
Les homonymies sur les titres de films ont constitué une complication significative.
Nous avons obtenu finalement 5 868 budgets supplémentaires qu'il a fallu retraiter :
- gestion des séparateurs décimaux différents,
- gestion des montants exprimés en unités ou en millions,
- gestion des fourchettes, 
- conversion des monnaies étrangères, y compris avant l'euro
- conversion en dollars constants de 2017 (avec l'indice d'inflation américain depuis 1960)


## Retraitement des variables cast et crew

La variable crew contient les noms de toute l’equipe de tournage. Elle nous permet de recuperer le nom du réalisateur.
De meme, la variable cast contient les noms de tous les acteurs du film .
/!\ jusqu’à 313 acteurs pour un même film !
on se limite aux 10 acteurs principaux.
l'information que nous souhaitions tirer du casting, c'est de savoir si les acteurs principaux et le realisateur etaient déja connus au moment de la sortie du film.
Nous avons donc recuperé par web scrapping sur wikipedia les recompenses obtenues pour chacun des 10 premiers acteurs de chaque film, et leur année d'obtention.
Nous distinguons les "small_awards" (petites recompenses) des "big_awards" ( prix internationaux prestigieux , comme une nomination aux oscars par exemple).
Ainsi, le nombre de prix déjà gagnés par les acteurs l'année de sortie du film sont a considérer comme une proxy de leur notoriété de l'epoque.
Nous créons 20 variables, correspondant aux nombre de petites et grandes recompenses des 10 acteurs principaux
De meme concernant le réalisateur, soit un total de 22 variables

A la fin des retraitements, nous avons 47 variables potentiellement explicatives.
""")

section_3 = pn.pane.Markdown("""
# 3. Statistiques descriptives
""")

section_4 = pn.Column(
    section_title,
    reg_title, reg0_title, reg0_text, reg0_img,
    reg1_title, reg1_text, reg1_img,
    reg2_title, reg2_text, reg2_img,
    reg3_title, reg3_text, reg3_img,
    reg4_title, reg4_text, reg4_img,
    reg5_title, reg5_text, reg5_img,
    reg6_title, reg6_text, reg6_img,
    reg7_title, reg7_text, reg7_img
    #, rf_title, rf_text, rf_img,
    # gb_title, gb_text, gb_img,
    # nn_title, nn_text, nn_img
)
section_4_old=pn.pane.Markdown("""
# 4. Machine Learning

## Modèles de régression

## Random Forest

Comme nous l'avons vu précédemment, le RMSE est plûtot bon, cependant le R² reste relativement faible. 
Etant donné que la distribution de la variable cible correspond à celle d'une loi normale, on peut supposer que le modèle prédit en moyenne correctement (bon RMSE) mais ne capte pas bien la variance des données (notes extrêmes).
Dans l'optimisation des hyperparamètres du modèle, nous allons donc chercher à maximiser le R² et regarder si cela améliore ou détériore le RMSE.

Les hyperparamètres que nous avons optimisés sont les suivants : 
- n_estimators : de 100 à 1000
- max_depth : de None à 100
- min_samples_split : de 2 à 10
- min_samples_leaf : de 1 à 4
- max_features : sqrt ou log2

L'optimisation des hyperparamètres s'est faite par cross validation sur 5 folds.

Le modèle Random Forest optimal est celui avec n_estimators=1000, max_depth = 50, min_samples_split=2, min_samples_leaf=2 et max_features=sqrt.
Ci-dessous le tableau des métriques classiques et pondérées pour le Random Forest :

Nous pouvons observer que toutes les métriques sont meilleures que celles de la régression linéaire. 
Les RMSE et MAE (classiques et pondérés) ont augmenté avec le R², ce qui indique que le modèle a réussi à mieux capturer la variance des données et à prédire plus finement les notes.
Les métriques pondérées sont toujours plus faibles que les métriques classiques, ce qui signifie que le modèle est meilleur sur les films avec le plus de votes, donc plus populaires.

## Gradient Boosting

Dans l'optimisation des hyperparamètres du modèle, nous allons chercher à maximiser le R² avec la même logique que pour le Random Forest.

Les hyperparamètres que nous avons optimisés sont les suivants : 
- n_estimators : de 100 à 1000
- learning rate : de 0.01 à 0.1
- max_depth : de 3 à 10
- min_samples_leaf : de 1 à 4
- subsample : de 0.8 à 1

L'optimisation des hyperparamètres s'est faite par cross validation sur 5 folds.

Le modèle Gradient Boosting optimal est celui avec n_estimators=1000, learning_rate=0.01, max_depth = 10, min_samples_leaf=3 et subsample=0.8.
Ci-dessous le tableau des métriques classiques et pondérées pour le Gradient Boosting :

## Réseaux de neurones
""")

section_5 = pn.pane.Markdown("""
# 5. Conclusion
Encore un autre contenu. Blablabla
""")

# ---- Zone centrale qui change ----
main_area = pn.Column(section_1, sizing_mode="stretch_width")

# ---- Fonctions de navigation ----
def show_section_1(event):
    main_area.objects = [section_1]

def show_section_2(event):
    main_area.objects = [section_2]

def show_section_3(event):
    main_area.objects = [section_3,
        histo_y,
        text_y_hist,
        histo_vote_count,
        text_vote_count_hist,
        graph_films_votes,
        texte_gr_films_votes,
        graph_films_notes,
        texte_gr_films_notes]

def show_section_4(event):
    main_area.objects = [section_4]

def show_section_5(event):
    main_area.objects = [section_5]

# ---- Sidebar = menu latéral ----
sidebar = pn.Column(
    pn.pane.Markdown("## Menu"),
    pn.widgets.Button(name="1. Introduction"),
    pn.widgets.Button(name="2. Retraitements"),
    pn.widgets.Button(name="3. Statistiques descriptives"),
    pn.widgets.Button(name="4. Machine Learning"),
    pn.widgets.Button(name="5. Conclusion"),
)

# Association des boutons aux fonctions
sidebar[1].on_click(show_section_1)
sidebar[2].on_click(show_section_2)
sidebar[3].on_click(show_section_3)
sidebar[4].on_click(show_section_4)
sidebar[5].on_click(show_section_5)

# ---- Template global ----
template = pn.template.MaterialTemplate(title="Projet Datascience - Cécile Prévot et Marie Le Pennec")

template.sidebar.append(sidebar)
template.main.append(main_area)

template.servable()


BootstrapTemplate
    [js_area] HTML(None, height=0, margin=0, sizing_mode='fixed', width=0)
    [actions] TemplateActions()
    [busy_indicator] LoadingSpinner(height=20, width=20)
    [2055967524616] Column
        [0] Markdown(str)
        [1] Button(name='1. Introduction')
        [2] Button(name='2. Retraitements')
        [3] Button(name='3. Statistiques d...)
        [4] Button(name='4. Machine Learning')
        [5] Button(name='5. Conclusion')
    [2055967878600] Column
        [0] Markdown(str)

In [9]:
#panel serve --autoreload --show --allow-websocket-origin=$(echo $VSCODE_PROXY_URI | cut -d '/' -f 3) /home/onyxia/work/formation_cepe/Projet/panel_projet.ipynb